In [21]:
import os
import zipfile
import pandas as pd
from google.colab import drive
from sklearn.model_selection import train_test_split
from collections import Counter
from PIL import Image

In [ ]:
# 1. 드라이브 마운트 & 압축 해제
drive.mount('/content/drive/')

zip_path = '/content/drive/MyDrive/오즈코딩/13해커톤/open.zip'
data_dir = '/content/chest_xray'

with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(data_dir)

In [23]:
# 2. CSV 로드
train_df = pd.read_csv(f'{data_dir}/train.csv')
test_df = pd.read_csv(f'{data_dir}/test.csv')
sample_submission = pd.read_csv(f'{data_dir}/sample_submission.csv')

In [24]:
# 3. CSV ↔ 실제 이미지 파일명 일치 확인
for name, df in [('train', train_df), ('test', test_df)]:
    files = set(os.listdir(f'{data_dir}/{name}'))
    print(f'{name}: CSV {len(df)}행 / 이미지 {len(files)}장 / 파일명 일치 {set(df["file_name"]) == files}')

train: CSV 5216행 / 이미지 5216장 / 파일명 일치 True
test: CSV 624행 / 이미지 624장 / 파일명 일치 True


In [25]:
# 4. 클래스 분포 (개수 + 비율)
dist = train_df['label'].value_counts().sort_index()
print(pd.DataFrame({'count': dist, 'ratio(%)': (dist / dist.sum() * 100).round(2)}))

       count  ratio(%)
label                 
0       1341     25.71
1       3875     74.29


In [30]:
image_dir = '/content/chest_xray/train'

size_counts = Counter()
mode_counts = Counter()

for file in train_df['file_name']:
    path = os.path.join(image_dir, file)

    with Image.open(path) as img:
        size_counts[img.size] += 1
        mode_counts[img.mode] += 1

print('이미지 크기별 개수:', size_counts)
print('이미지 모드별 개수:', mode_counts)

이미지 크기별 개수: Counter({(224, 224): 5216})
이미지 모드별 개수: Counter({'L': 5216})


In [31]:
# 5. Train / Validation 분할 (label 비율 유지)
train_split, val_split = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df['label']
)
print(f'Train: {len(train_split)} / Validation: {len(val_split)} / Test: {len(test_df)}')

Train: 4172 / Validation: 1044 / Test: 624
